### This is an example of a portfolio of shares traded on other markets than USA. ###

We must make sure that the business calendar and the price time series are consistent with the exchange. 
Also any price/cash value is assumed to be denominated in the exchange currency. 

We use Stockholm exchange as an example.

In [1]:
import numpy as np
import azapy as az

print(f"azapy version {az.version()} >= 1.2.5")

azapy version 1.2.6 >= 1.2.5


Let's start by setting the exchange business calendar.

The Stockholm business calendar is abbreviated as "XSTO"

In [2]:
exch_calendar = 'XSTO'

Let's specify a set of (popular) shares traded on the Stockholm exchange.

Note the suffix .ST is specific to yahoo market data provider.

In [3]:
symb = ['GETI-B.ST', 'AZN.ST', 'SWED-A.ST', 'VOLV-B.ST', 'ASSA-B.ST', 'ELUX-B.ST']


Collect market data starting from 2012-01-01

In [4]:
sdate = "2012-01-01"
edate = "today"

mktdir = "../MkTdata"

mktdata = az.readMkT(symb, sdate=sdate, edate=edate, file_dir=mktdir, calendar=exch_calendar)

read GETI-B.ST data from file
get GETI-B.ST updates from yahoo
save GETI-B.ST updated data to file
read AZN.ST data from file
get AZN.ST updates from yahoo
save AZN.ST updated data to file
read SWED-A.ST data from file
get SWED-A.ST updates from yahoo
save SWED-A.ST updated data to file
read VOLV-B.ST data from file
get VOLV-B.ST updates from yahoo
save VOLV-B.ST updated data to file
read ASSA-B.ST data from file
get ASSA-B.ST updates from yahoo
save ASSA-B.ST updated data to file
read ELUX-B.ST data from file
get ELUX-B.ST updates from yahoo
save ELUX-B.ST updated data to file

Request between 2012-01-02 : 2026-05-05
              GETI-B.ST      AZN.ST   SWED-A.ST   VOLV-B.ST   ASSA-B.ST  \
source            yahoo       yahoo       yahoo       yahoo       yahoo   
force             False       False       False       False       False   
save               True        True        True        True        True   
file_dir     ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata   

Note the value 'Yes' for the field 'error'. It means that collected market data has missing values. This could be a major concern for further computations.

Let's look more closely at the missing data. Instead of using the function `readMkT as above, we will use the class `MkTreader` to upload the historical data. 
This has the advantage of providing more info about the missing data.

In [5]:
mkt = az.MkTreader()
mktdata_ = mkt.get(symb, sdate=sdate, edate=edate, file_dir=mktdir, calendar=exch_calendar)

read GETI-B.ST data from file
read AZN.ST data from file
read SWED-A.ST data from file
read VOLV-B.ST data from file
read ASSA-B.ST data from file
read ELUX-B.ST data from file

Request between 2012-01-02 : 2026-05-05
              GETI-B.ST      AZN.ST   SWED-A.ST   VOLV-B.ST   ASSA-B.ST  \
source            yahoo       yahoo       yahoo       yahoo       yahoo   
force             False       False       False       False       False   
save               True        True        True        True        True   
file_dir     ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata   
file_format         csv         csv         csv         csv         csv   
api_key            None        None        None        None        None   
nrow               3593        3593        3593        3593        3593   
sdate        2012-01-02  2012-01-02  2012-01-02  2012-01-02  2012-01-02   
edate        2026-05-05  2026-05-05  2026-05-05  2026-05-05  2026-05-05   
error               Yes         

So far we have replicated the `readMkT` functionality; `mktdata` and `mktdata_` are identical.

However, the `mkt` object allows us to enquire about the nature of the errors.

In [6]:
mkt.get_error_log()

{'GETI-B.ST': {'mid': DatetimeIndex(['2013-05-10', '2019-06-10', '2019-06-17', '2019-08-05'], dtype='datetime64[us]', freq=None)},
 'AZN.ST': {'mid': DatetimeIndex(['2013-05-10', '2019-06-10', '2019-06-17', '2019-08-05'], dtype='datetime64[us]', freq=None)},
 'SWED-A.ST': {'mid': DatetimeIndex(['2013-05-10', '2019-06-10', '2019-06-17', '2019-08-05'], dtype='datetime64[us]', freq=None)},
 'VOLV-B.ST': {'mid': DatetimeIndex(['2013-05-10', '2019-06-10', '2019-06-17', '2019-08-05'], dtype='datetime64[us]', freq=None)},
 'ASSA-B.ST': {'mid': DatetimeIndex(['2013-05-10', '2019-06-10', '2019-06-17', '2019-08-05'], dtype='datetime64[us]', freq=None)},
 'ELUX-B.ST': {'mid': DatetimeIndex(['2013-05-10', '2019-06-10', '2019-06-17', '2019-08-05'], dtype='datetime64[us]', freq=None)}}

It appears that the missing data occurs in the middle of the time series, for the same 4 dates across all symbols ('2013-05-10', '2019-06-10', '2019-06-17', '2019-08-05').

Note, this behavior was recorded at the time when we were writing this script. In the future, yahoo may choose to correct these errors. 
In that case the next few lines of code involving data imputation may become superfluous. 

Since we are dealing with a very small number of missing data located well in the past and unrelated with any special market critical events, we can 
proceed to fill in the missing data using a linear imputation.

In [7]:
mktdata2 = mkt.set_imputation(method='linear')

`mktdata2` contain the same information as `mktdata` except that the missing data were replaced with interpolated values. 

It is easy to verify this by using the `summary_MkTdata` function

In [8]:
az.summary_MkTdata(mktdata2, calendar=exch_calendar)

,symbol,begin,end,length,na_total,na_b,na_e,cont
0,ASSA-B.ST,2012-01-02,2026-05-05,3597,0,0,0,0
1,AZN.ST,2012-01-02,2026-05-05,3597,0,0,0,0
2,ELUX-B.ST,2012-01-02,2026-05-05,3597,0,0,0,0
3,GETI-B.ST,2012-01-02,2026-05-05,3597,0,0,0,0
4,SWED-A.ST,2012-01-02,2026-05-05,3597,0,0,0,0
5,VOLV-B.ST,2012-01-02,2026-05-05,3597,0,0,0,0


Note the zero values in the last 4 columns.

For example the uncorrected market data, 'mktdata' from the original extraction (without imputation), returns

In [9]:
az.summary_MkTdata(mktdata, calendar=exch_calendar)

,symbol,begin,end,length,na_total,na_b,na_e,cont
0,ASSA-B.ST,2012-01-02,2026-05-05,3593,0,0,0,4
1,AZN.ST,2012-01-02,2026-05-05,3593,0,0,0,4
2,ELUX-B.ST,2012-01-02,2026-05-05,3593,0,0,0,4
3,GETI-B.ST,2012-01-02,2026-05-05,3593,0,0,0,4
4,SWED-A.ST,2012-01-02,2026-05-05,3593,0,0,0,4
5,VOLV-B.ST,2012-01-02,2026-05-05,3593,0,0,0,4


It signals (us we already know) the fact that 4 working dates, relative to the `exch_calendar` business calendar, are missing.

A faster way to compute the imputed market data is to use the `readMkT` with `imputation` value set.

In [10]:
mktdata2_ = az.readMkT(symb, sdate=sdate, edate=edate, file_dir=mktdir, calendar=exch_calendar, imputation='linear')

read GETI-B.ST data from file
read AZN.ST data from file
read SWED-A.ST data from file
read VOLV-B.ST data from file
read ASSA-B.ST data from file
read ELUX-B.ST data from file

Request between 2012-01-02 : 2026-05-05
              GETI-B.ST      AZN.ST   SWED-A.ST   VOLV-B.ST   ASSA-B.ST  \
source            yahoo       yahoo       yahoo       yahoo       yahoo   
force             False       False       False       False       False   
save               True        True        True        True        True   
file_dir     ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata  ../MkTdata   
file_format         csv         csv         csv         csv         csv   
api_key            None        None        None        None        None   
nrow               3593        3593        3593        3593        3593   
sdate        2012-01-02  2012-01-02  2012-01-02  2012-01-02  2012-01-02   
edate        2026-05-05  2026-05-05  2026-05-05  2026-05-05  2026-05-05   
error               Yes         

mktdata2_ is identical to mktdata2. The missing data was filled using linear interpolation. 

Note:
1. The printed output (`verbose=True`) refers to the raw data (before imputation). Therefore, the field `error` is still `Yes`. This reminds us
that some of the market data was filled in. The `summary_MkTdata(mktdata2_, calendar=exch_calendar)` will summarize the corrected version of the market data. In general 
any imputation algorithm will introduce bias; therefore it is advisable to have a deep understanding of the nature of missing data and how it may influence future calculations.
If it is possible, it is always better to get the market data without omissions. In any event, before 
applying an imputation mechanism it is better to have a 
2. The saved market data (in `file_dir`) it always the raw (uncorrected) market data.

Further we will use `mktdata2` in our computations.

We will set a mixture CVaR risk measure and we will perform a backtesting for maximum Sharpe ratio optimal portfolio quarterly rebalanced. 

Note: The exchange calendar must be passed to the backtest simulator, in our case `Port_CVaR` object constructor.

In [11]:
alpha = [0.95, 0.90, 0.85]
coef = [1, 1, 1]

pr2 = az.Port_CVaR(mktdata2, pname='CVaRPort', capital=1000000, calendar=exch_calendar)
port2 = pr2.set_model(alpha=alpha, coef=coef, hlength=1.25)

The rest is the same as for any backtest analysis (e.g. see `OutOfSample_example.jpynb`). The monetary values should be understood to be denominated in the exchange currency. In our case 
is Swedish Krona. 

In [12]:
_ = pr2.port_view(fancy=True, title="Portfolio performance")

In [13]:
_ = pr2.port_view_all(fancy=True, title="Relative performance")

In [14]:
pr2.get_weights()

,Droll,Dfix,ASSA-B.ST,AZN.ST,ELUX-B.ST,GETI-B.ST,SWED-A.ST,VOLV-B.ST
0,2015-06-25,2015-06-24,0.570651,0.000000,0.429349,0.000000,0.000000e+00,0.000000
1,2015-09-25,2015-09-24,0.756710,0.000000,0.243290,0.000000,0.000000e+00,0.000000
2,2015-12-23,2015-12-22,0.706352,0.000000,0.135443,0.158205,0.000000e+00,0.000000
3,2016-03-24,2016-03-23,1.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
4,2016-06-27,2016-06-23,1.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
5,2016-09-27,2016-09-26,1.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000
6,2016-12-27,2016-12-23,0.246507,0.000000,0.000000,0.000000,0.000000e+00,0.753493
7,2017-03-28,2017-03-27,0.132503,0.034709,0.204455,0.000000,3.896136e-01,0.238720
8,2017-06-27,2017-06-26,0.128135,0.081097,0.094026,0.000000,5.595450e-01,0.137197
9,2017-09-26,2017-09-25,0.000000,0.000000,0.426531,0.000000,5.634383e-01,0.010030


In [15]:
pr2.port_perf(fancy=True)

,RR,DD,RoMaD,DD_date,DD_start,DD_end,DD_days
symbol,,,,,,,
CVaRPort,13.11,-34.30,0.382143,2025-04-07,2024-03-27,2025-12-11,624
AZN.ST,13.38,-29.95,0.446848,2025-04-09,2024-08-29,2025-11-25,453
ASSA-B.ST,15.47,-35.35,0.437578,2020-03-23,2020-02-20,2021-03-25,399
SWED-A.ST,16.62,-49.11,0.338420,2020-05-14,2018-09-27,2021-10-21,1120
VOLV-B.ST,14.65,-43.81,0.334505,2020-03-18,2020-02-20,2020-09-28,221
GETI-B.ST,4.98,-59.53,0.083623,2024-11-21,2021-11-15,NaN,1632
ELUX-B.ST,0.10,-79.09,0.001207,2026-04-24,2021-03-25,NaN,1867


In [16]:
pr2.port_drawdown(fancy=True)

,DD,Date,Start,End,NrDays
No,,,,,
1,-34.30,2025-04-07,2024-03-27,2025-12-11,624
2,-30.66,2020-03-16,2020-01-17,2021-03-22,430
3,-24.18,2022-03-08,2021-11-17,2023-06-16,576
4,-22.71,2016-02-11,2015-11-30,2017-03-13,469
5,-19.31,2018-06-26,2017-10-03,2018-09-27,359


In [17]:
pr2.port_annual_returns(fancy=True)

,CVaRPort
year,
2015,3.33%
2016,-4.89%
2017,20.70%
2018,4.76%
2019,37.58%
2020,-15.91%
2021,68.20%
2022,-7.62%
2023,23.54%


In [18]:
pr2.port_monthly_returns(fancy=True)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
month,,,,,,,,,,,,
1,nan%,-2.67%,3.31%,4.97%,-0.21%,-1.29%,10.78%,-2.13%,1.56%,-3.68%,4.83%,7.97%
2,nan%,-5.09%,5.59%,-1.50%,15.07%,-11.03%,0.51%,-10.98%,2.66%,11.39%,-2.24%,0.32%
3,nan%,-5.89%,8.76%,-2.75%,-2.75%,-0.13%,10.58%,0.49%,-2.70%,1.70%,-14.42%,0.52%
4,nan%,5.12%,4.39%,-2.00%,-0.36%,5.19%,11.70%,1.17%,3.83%,-1.22%,-0.10%,1.51%
5,nan%,2.97%,1.31%,1.71%,-5.63%,-5.42%,2.55%,-2.24%,-0.67%,-2.60%,5.29%,0.56%
6,-2.68%,-2.76%,1.28%,-2.01%,9.26%,-4.53%,7.95%,0.61%,5.87%,-1.17%,1.84%,nan%
7,5.02%,9.12%,-0.95%,7.73%,5.07%,13.37%,2.40%,3.21%,1.68%,6.15%,4.94%,nan%
8,-5.94%,-7.67%,1.03%,2.16%,3.37%,-6.20%,-4.15%,-3.21%,-1.29%,-1.46%,2.12%,nan%
9,-3.68%,5.92%,6.09%,5.67%,1.17%,4.38%,3.88%,-7.00%,1.40%,0.56%,5.67%,nan%


In [19]:
pr2.port_period_returns(fancy=True)

,Droll,Dfix,RR,ASSA-B.ST,AZN.ST,ELUX-B.ST,GETI-B.ST,SWED-A.ST,VOLV-B.ST
0,2015-06-25,2015-06-24,-9.15,57.07,0.00,42.93,0.00,0.00,0.00
1,2015-09-25,2015-09-24,12.30,75.67,0.00,24.33,0.00,0.00,0.00
2,2015-12-23,2015-12-22,-9.97,70.64,0.00,13.54,15.82,0.00,0.00
3,2016-03-24,2016-03-23,-1.17,100.00,0.00,0.00,0.00,0.00,0.00
4,2016-06-27,2016-06-23,4.70,100.00,0.00,0.00,0.00,0.00,0.00
5,2016-09-27,2016-09-26,3.97,100.00,0.00,0.00,0.00,0.00,0.00
6,2016-12-27,2016-12-23,18.15,24.65,0.00,0.00,0.00,0.00,75.35
7,2017-03-28,2017-03-27,6.29,13.25,3.47,20.45,0.00,38.96,23.87
8,2017-06-27,2017-06-26,5.30,12.81,8.11,9.40,0.00,55.95,13.72
9,2017-09-26,2017-09-25,-5.51,0.00,0.00,42.65,0.00,56.34,1.00


In [20]:
pr2.get_nshares()

,ASSA-B.ST,AZN.ST,ELUX-B.ST,GETI-B.ST,SWED-A.ST,VOLV-B.ST,_CASH_
Droll,,,,,,,
2015-06-25,3551,0,1940,0,0,0,0
2015-09-25,4600,0,1199,0,0,0,0
2015-12-23,4077,0,822,1068,0,0,0
2016-03-24,5589,0,0,0,0,0,0
2016-06-27,5476,0,0,0,0,0,0
2016-09-27,5770,0,0,0,0,0,0
2016-12-27,1431,0,0,0,0,6933,0
2017-03-28,851,72,1215,0,2081,2144,0
2017-06-27,844,168,497,0,3465,1171,0


In [21]:
pr2.get_account(fancy=True)

,ASSA-B.ST,AZN.ST,ELUX-B.ST,GETI-B.ST,SWED-A.ST,VOLV-B.ST,_CASH_,cash_invst,cash_roll,cash_divd
Droll,,,,,,,,,,
2015-06-25,3551.0,0.0,1940.0,0.0,0.0,0.0,0.0,996780.22,3219.78,0.00
2015-09-25,4600.0,0.0,1199.0,0.0,0.0,0.0,0.0,924346.89,-15527.34,0.00
2015-12-23,4077.0,0.0,822.0,1068.0,0.0,0.0,0.0,1023887.06,-22463.91,0.00
2016-03-24,5589.0,0.0,0.0,0.0,0.0,0.0,0.0,873001.78,20065.62,0.00
2016-06-27,5476.0,0.0,0.0,0.0,0.0,0.0,0.0,886564.37,53171.17,14810.85
2016-09-27,5770.0,0.0,0.0,0.0,0.0,0.0,0.0,998210.00,2309.17,0.00
2016-12-27,1431.0,0.0,0.0,0.0,0.0,6933.0,0.0,993035.70,-5787.54,0.00
2017-03-28,851.0,72.0,1215.0,0.0,2081.0,2144.0,0.0,1164262.46,-6381.19,0.00
2017-06-27,844.0,168.0,497.0,0.0,3465.0,1171.0,0.0,1261978.80,5492.13,36990.20
